---
title: Глава 8. Группировка и агрегация
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-08-29
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Инструмент сборки статических сайтов
    JupySQL: Расширение для запуска и подсветки SQL в Jupyter
    GitHub: Платформа хостинга репозиториев и совместной разработки
    GitHub Pages: Сервис бесплатного хостинга статических сайтов
    GitHub Actions: Платформа автоматизации рабочих процессов и CI/CD
    Pandas: Библиотека Python для анализа и обработки данных
    Polars: Мощный аналог Pandas на Rust/Python
---

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="*UHB5rdx",
)

engine = create_engine(connection_url)

%load_ext sql

%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False
%config SqlMagic.autopandas = True

%sql engine

print(f'Pandas ver. {pd.__version__}')
print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

Pandas ver. 3.0.5
SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


## Концепция группировки

При наличии 599 клиентов, имеющих более 16000 записей об аренде, просмотрев необработанные данные, невозможно определить какие клиенты взяли напрокат больше всего фильмов:

In [3]:
%%sql
SELECT customer_id FROM rental;

16044 rows affected.

,customer_id
0,1
1,1
2,1
3,1
4,1
...,...
16039,599
16040,599
16041,599
16042,599


Вместо этого мы можем попросить сервер базы данных сгруппировать данные с помощью предложения `group_by` для группировки данных о прокате по идентификатору клиента:

In [4]:
%%sql
SELECT customer_id
FROM rental
GROUP BY customer_id;

599 rows affected.

,customer_id
0,1
1,2
2,3
3,4
4,5
...,...
594,595
595,596
596,597
597,598


Результирующий набор содержит по одной строке для каждого отдельного значения в столбце customer_id. Причина меньшего результирующего набора в том, что некоторые клиенты брали напрокат более одного фильма.

Чтобы увидеть, сколько фильмов было взято напрокат каждым клиентом, можно использовать _**агрегатную функцию**_ `count(*)` в предложении select для подсчета количества строк в каждой группе:

In [6]:
%%sql
SELECT
    customer_id,
    COUNT(*) AS rental_count
FROM rental
GROUP BY customer_id;

599 rows affected.

,customer_id,rental_count
0,1,32
1,2,27
2,3,26
3,4,22
4,5,38
...,...,...
594,595,30
595,596,28
596,597,25
597,598,22


:::{attention} Агрегатная функция `count()`
:class: simple
:icon: false
Подсчитывает количество строк в каждой группе. Звездочка сообщает серверу, что нужно считать все что есть в группе.
```sql
COUNT(*)
COUNT(имя_колонки)
COUNT(DISTINCT имя_колонки)
```
Результат зависит от аргумента в скобках
- `COUNT(*)` — производит подсчет **абсолютно всех физических строк** в группе. Не важно, что находится внутри ячеек: он учитывает строки, полностью состоящие из NULL-значений, и дубликаты.
- `COUNT(имя_колонки)` — считает количество **заполненных значений** в конкретной колонке. Игнорирует (пропускает) значения NULL. Если в группе из 10 строк у двух строк в этой колонке стоит NULL, функция вернет 8.
- `COUNT(DISTINCT имя_колонки)` — считает только **уникальные** заполненные значения в колонке. Так же игнорирует NULL, а все одинаковые значения считает за один раз.
:::

Чтобы определить, какие клиенты взяли напрокат больше всего фильмов, добавим предложение `order by`:

In [7]:
%%sql
SELECT
    customer_id,
    COUNT(*) AS rental_count
FROM rental
GROUP BY customer_id
ORDER BY 2 DESC;

599 rows affected.

,customer_id,rental_count
0,148,46
1,526,45
2,144,42
3,236,42
4,75,41
...,...,...
594,248,15
595,61,14
596,110,14
597,281,14


При группировке данных может потребоваться отфильтровать нежелательные данные из набора результатов на основе групп данных (а не необработанных данных). Поскольку предложение `group by` выполняется после того, как вычислено предложение `where`, добавить условия фильтрации к предложению `where` для этой цели нельзя.

Например, вот к чему приводит попытка отфильтровать клиентов, бравших напрокат менее 40 фильмов:

In [9]:
%%sql
SELECT
    customer_id,
    COUNT(*)
FROM rental
WHERE COUNT(*) >= 40
GROUP BY customer_id;

RuntimeError: (pymysql.err.ProgrammingError) (1111, 'Invalid use of group function')
[SQL: SELECT
    customer_id,
    COUNT(*)
FROM rental
WHERE COUNT(*) >= 40
GROUP BY customer_id;]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [11]:
%config SqlMagic.autopandas = False

Вы не можете обратиться к агрегатной функции `count(*)` в предложении `where` потому что во время вычисления предложения `where` группы еще не были сгенерированы. Вместо этого вы должны поместить условия группового фильтра в предложение `having`:

In [12]:
%%sql
SELECT
    customer_id,
    COUNT(*)
FROM rental
GROUP BY customer_id
HAVING COUNT(*) >= 40;

7 rows affected.

customer_id,COUNT(*)
75,41
144,42
148,46
197,40
236,42
469,40
526,45


Чтобы отправить лучшим клиентам купоны на бесплатную аренду, осталось воспользоваться предыдущей Главой 5 для вывода имени клиента и его адреса:

In [6]:
%%sql
SELECT 
    c.customer_id,
    c.first_name,
    c.last_name,
    a.address,
    ci.city,
    COUNT(*) AS total_rentals
FROM rental AS r
    INNER JOIN customer AS c ON r.customer_id = c.customer_id
    INNER JOIN address AS a ON c.address_id = a.address_id
    INNER JOIN city AS ci ON a.city_id = ci.city_id
GROUP BY 
    c.customer_id,
    c.first_name,
    c.last_name,
    a.address,
    ci.city
HAVING COUNT(*) >= 40;

7 rows affected.

customer_id,first_name,last_name,address,city,total_rentals
75,TAMMY,SANDERS,1551 Rampur Lane,Changhwa,41
144,CLARA,SHAW,1027 Songkhla Manor,Molodetšno,42
148,ELEANOR,HUNT,1952 Pune Lane,Saint-Denis,46
197,SUE,PETERS,817 Bradford Loop,Changzhou,40
236,MARCIA,DEAN,1479 Rustenburg Boulevard,Tanza,42
469,WESLEY,BULL,1469 Plock Lane,Ourense (Orense),40
526,KARL,SEAL,1427 Tabuk Place,Cape Coral,45


::::{tip} Пояснение
:class: dropdown simple
:open: false

В блоке `GROUP BY` перечислены все неагрегированные колонки, которые указали в `SELECT`:
- Согласно строгому стандарту SQL, если вы выводите в `SELECT` текстовые поля (имя или адрес), СУБД должна четко понимать, как их группировать.
- Хотя `customer_id` является уникальным ключом и MySQL 8.0 достаточно *умен*, чтобы понять, что у одного ID не может быть двух разных имен, хорошим тоном и гарантией защиты от ошибок является явное перечисление всех выводимых колонок в блоке `GROUP BY`.
- Когда вы пишете `GROUP BY customer_id, first_name, last_name, address, city`, для сервера это означает: Создай кучку только тогда, когда совпадает ВСЕ указанные признаки одновременно. Но поскольку имя, фамилия и адрес намертво привязаны к конкретному `customer_id` (у клиента №1 всегда одно и то же имя, фамилия и адрес), то состав этих кучек физически вообще не изменится!

:::{card}
Мы перечисляем эти поля в `GROUP BY` исключительно ради того, чтобы удовлетворить строгое требование SQL-движка: Всё, что ты выводишь на экран в `SELECT`, должно быть либо внутри агрегатной функции, либо зафиксировано в правилах группировки `GROUP BY`.
:::

SQL-движок видит, что `customer_id` уникален для каждого человека. Из-за этого добавление в `GROUP BY` зависимых полей first_name, address и т.д. физически не может раздробить группу сильнее. Движок группирует по совокупности этих полей, но *ведущим* (определяющим размер кучки) фактором остается `customer_id`.

---
Что и как считает `COUNT(*)`? \
Порядок выполнения запроса в SQL выглядит так: `FROM` (включая все `JOIN`) ➔ `GROUP BY` ➔ `SELECT / COUNT`.
- **До группировки**: сначала SQL-движок выполняет все `INNER JOIN`. Он берет каждую строчку аренды из таблицы `rental`, подклеивает к ней имя клиента, затем адрес, затем город. На выходе получается одна огромная *плоская* временная таблица, где строк ровно столько же, сколько было в `rental` (ведь у каждой аренды есть один клиент, один адрес и один город). Каждый факт аренды превратился в длинную строку.
- В момент группировки: Сервер берет эту огромную соединенную таблицу и раскладывает её на кучки по нашему правилу из первого пункта (по клиентам).
- В момент подсчета: функция `COUNT(*)` расшифровывается как *посчитай количество физических строк в получившейся кучке*.

---
`GROUP BY` по нескольким полям в данном случае не дробит группы сильнее, а просто *легализует* вывод этих полей в SELECT.

`COUNT(*)` после JOIN считает количество строк в финальной сгруппированной таблице. Поскольку связь была *один ко многим* (`customer` к `rental`), количество строк в группе всё так же равно количеству аренд клиента.
::::

В таблице `rental` колонка `return_date` содержит дату и время возврата диска. Если покупатель еще не принес фильм обратно в прокат, в эту ячейку записывается `NULL`.

Hапишем запрос, который сгруппирует данные по сотрудникам проката `staff_id` и покажет: сколько всего фильмов выдал каждый сотрудник, сколько фильмов ему уже вернули и сколько до сих пор находится на руках у клиентов.

In [12]:
%%sql
SELECT
    staff_id,
    COUNT(*) AS total_rents,
    COUNT(return_date) AS returned_rentals,
    (COUNT(*) - COUNT(return_date)) AS still_rented
FROM rental
GROUP BY staff_id;

2 rows affected.

staff_id,total_rents,returned_rentals,still_rented
1,8040,7955,85
2,8004,7906,98


## Агрегатные функции

Агрегатные функции выполняют определенные операции над всеми строками в группе. Распространенные агрегатные функции:
| Функция | Описание                                  |
| ------- | ----------------------------------------- |
| max()   | Возвращает максимальное значение в наборе |
| min()   | Возвращает минимальное значение в наборе  |
| avg()   | Возвращает усредненное значение в наборе  |
| sum()   | Возвращает сумму значений в наборе        |
| count() | Возвращает количество значений в наборе   |

Вот как выглядит запрос, в котором используются все распространенные агрегатные функции для анализа данных по прокату фильмов:

In [13]:
%%sql
SELECT
    MAX(amount) max_amt,
    MIN(amount) min_amt,
    AVG(amount) avg_amt,
    SUM(amount) tot_amt,
    COUNT(*) num_payments
FROM payment;

1 rows affected.

max_amt,min_amt,avg_amt,tot_amt,num_payments
11.99,0.00,4.201356,67406.56,16044


### Неявная и явная группировка

В предыдущем примере каждое значение, возвращаемое запросом, генерируется агрегатной функцией. Поскольку предложения `group by` в запросе нет, существует единственная неявная группа (все строки в таблице payment).

Однако в большинстве случаев требуется получить дополнительные столбцы вместе со столбцами, генерируемыми агрегатными функциями. Например, можно расширить предыдущий запрос не для всех клиентов одновременно, а для каждого клиента.

Для каждого такого запроса нужно вместе с пятью агрегатными функциями выполнить выборку customer_id:

In [6]:
%%sql
SELECT
    customer_id,
    MAX(amount) max_amt,
    MIN(amount) min_amt,
    AVG(amount) avg_amt,
    SUM(amount) tot_amt,
    COUNT(*) num_payments
FROM payment;

RuntimeError: (pymysql.err.OperationalError) (1140, "In aggregated query without GROUP BY, expression #1 of SELECT list contains nonaggregated column 'sakila.payment.customer_id'; this is incompatible with sql_mode=only_full_group_by")
[SQL: SELECT
    customer_id,
    MAX(amount) max_amt,
    MIN(amount) min_amt,
    AVG(amount) avg_amt,
    SUM(amount) tot_amt,
    COUNT(*) num_payments
FROM payment;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


Однако, мы получим сообщение об ошибке.

Этот запрос не выполняется, потому что в нем не указано *явно* как должны быть сгруппированы данные. Следовательно, в запрос нужно добавить предложение `group by` чтобы указать к какой группе строк следует применять агрегатные функции:

In [7]:
%%sql
SELECT
    customer_id,
    MAX(amount) max_amt,
    MIN(amount) min_amt,
    AVG(amount) avg_amt,
    SUM(amount) tot_amt,
    COUNT(*) num_payments
FROM payment
GROUP BY customer_id;

599 rows affected.

,customer_id,max_amt,min_amt,avg_amt,tot_amt,num_payments
0,1,9.99,0.99,3.708750,118.68,32
1,2,10.99,0.99,4.767778,128.73,27
2,3,10.99,0.99,5.220769,135.74,26
3,4,8.99,0.99,3.717273,81.78,22
4,5,9.99,0.99,3.805789,144.62,38
...,...,...,...,...,...,...
594,595,10.99,0.99,3.923333,117.70,30
595,596,6.99,0.99,3.454286,96.72,28
596,597,8.99,0.99,3.990000,99.75,25
597,598,7.99,0.99,3.808182,83.78,22


При включении предложения `group by` сервер понимает, что надо сначала сгруппировать строки, имеющие одинаковое значение в столбце customer_id, а затем применить к каждой из 599 групп агрегатные функции.

#### Подсчет различных значений

При использовании функции `count()` для определения количества членов в каждой группе у вас есть выбор: подсчитать *все* элементы группы или подсчитать только *различные* значения столбца среди всех жлементов группы.

Рассмотрим пример, в котором функция `count()` и столбец customer_id используются двумя разными способами:

In [14]:
%%sql
SELECT
    COUNT(customer_id) num_rows,
    COUNT(DISTINCT customer_id) num_customers
FROM payment;

1 rows affected.

num_rows,num_customers
16044,599


При указании ключевого слова `distinct` функция `count()` проверяет значения столбца для каждого члена группы, находя и удаляя дубликаты, а не просто подсчитывает количество значений в группе.

### Использование выражений

Наряду с использованием столбцов в качестве аргументов агрегатных функций можно исопользовать и выражения.

Например, можно найти максимальное количество дней между моментом, когда фильм был взят напрокат и последующим его возвратом:

In [16]:
%%sql
SELECT MAX(DATEDIFF(return_date, rental_date))
FROM rental;

1 rows affected.

"MAX(DATEDIFF(return_date, rental_date))"
10


Функция `datediff()` используется для вычисления количества дней между датой возврата фильма и датой взятия его напрокат для каждой аренды фильма, а функция `max()` возвращает максимальное найденное значение.

### Обработка значений *null*

При выполнении агрегации (на самом деле – любого вида числовых вычислений) всегда следует учитывать как на результат вычислений могут повлиять значения `null`.